# 임베딩과 의미 유사도

단어가 다르더라도 뜻이 비슷한 문장을 숫자로 비교할 수 있을까요?

Python 3.12와 잠긴 패키지를 사용합니다. 첫 실행에서는 로컬 모델을 다운로드합니다.

## 다국어 문장 임베딩

문장 네 개가 **입력**입니다. `model.encode()`는 각 문장을 384개의 실수로 바꾸어 NumPy 배열로 **반환**합니다. `shape`의 `(4, 384)`는 4개 행이 문장 네 개에, 384개 열이 각 문장의 숫자 표현에 대응한다는 뜻입니다. `vectors[0, :5]`는 구조를 확인하려고 첫 문장의 앞 다섯 값만 고릅니다.

In [ ]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_ID, revision="e8f8c211226b894fcb81acc59f3b34ba3efd5f42")
sentences = [
    "주문한 상품을 취소하고 싶습니다.",
    "구매를 취소하려면 어떻게 하나요?",
    "내일 서울은 비가 옵니다.",
    "I want to cancel my order.",
]
vectors = model.encode(sentences, normalize_embeddings=True)
print("shape:", vectors.shape)
print("첫 문장 앞 5개 값:", np.round(vectors[0, :5], 4))

## 두 숫자 벡터로 표기 읽기

`[3, 4]`와 `[4, 3]`을 두 행으로 담습니다. `shape`는 행과 열의 수를 반환합니다. `np.linalg.norm(..., axis=1)`은 각 행의 길이를 계산하고, 각 행을 그 길이로 나누면 길이 1인 정규화 벡터가 됩니다. `T`로 행과 열을 바꾼 뒤 `@`를 적용하면 모든 행 쌍의 점곱 표가 반환됩니다.

In [ ]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
toy = np.array([[3.0, 4.0], [4.0, 3.0]])
lengths = np.linalg.norm(toy, axis=1, keepdims=True)
normalized_toy = toy / lengths
toy_scores = normalized_toy @ normalized_toy.T
print("shape:", toy.shape)
print("lengths:", lengths.ravel())
print("normalized:\n", normalized_toy)
print("cosine table:\n", toy_scores)

## 코사인 유사도 비교

실제 문장 벡터는 길이 1로 정규화되어 있습니다. `vectors.T`는 행과 열을 바꾸고, `@`는 행렬 곱을 수행합니다. 따라서 `vectors @ vectors.T`는 모든 문장 쌍의 점곱, 곧 코사인 유사도를 4×4 배열로 반환합니다. 출력의 `0 vs i`는 항상 0번 취소 문장을 기준으로 읽습니다.

In [ ]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
scores = vectors @ vectors.T
for i in range(1, len(sentences)):
    print(f"0 vs {i}: {scores[0, i]:.4f} | {sentences[i]}")
nearest = np.argsort(-scores[0])[1:]
print("\n유사도 순서:", nearest.tolist())

## 단어 하나 바꾸기

'취소'를 '반품'으로 바꿔 유사도가 어떻게 달라지는지 봅니다.

In [ ]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
changed = model.encode(["주문한 상품을 반품하고 싶습니다."], normalize_embeddings=True)[0]
print("취소 문장 vs 반품 문장:", round(float(vectors[0] @ changed), 4))
print("취소 문장 vs 날씨 문장:", round(float(vectors[0] @ vectors[2]), 4))

## 구조 검증

예상한 실험 조건과 출력 구조가 유지되었는지 확인합니다.

In [ ]:
assert toy.shape == (2, 2) and np.allclose(lengths.ravel(), [5.0, 5.0])
assert np.allclose(np.diag(toy_scores), [1.0, 1.0])
assert vectors.shape == (4, 384) and np.isfinite(vectors).all()
assert scores[0, 3] > scores[0, 1] > scores[0, 2]
assert float(vectors[0] @ changed) > scores[0, 2]
print("검증: 2차원 표기, 384차원 벡터 구조와 본문의 유사도 순서를 확인했습니다.")